In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
=======================================================
# 1. CONFIGURAÇÃO
EMB_PATH = Path("../data/processed/08.FRAIL_with_tsvd_10dim.csv")
LLM_PATH = Path("../data/processed/09.llm.csv")

# 2. CARREGAMENTO E MERGE
print("--- [PASSO 1] A CARREGAR E A FUNDIR DADOS ---")

emb = pd.read_csv(EMB_PATH) 
llm = pd.read_csv(LLM_PATH)

# Garantir tipos para chaves de fusão
for k in ["ID_Aluno", "Momento"]:
    emb[k] = pd.to_numeric(emb[k], errors="coerce").astype("Int64")
    llm[k] = pd.to_numeric(llm[k], errors="coerce").astype("Int64")

# Remover duplicados exatos dentro de cada ficheiro antes do merge
emb = emb.sort_values(["ID_Aluno","Momento"]).drop_duplicates(subset=["ID_Aluno","Momento"], keep="first")
llm = llm.sort_values(["ID_Aluno","Momento"]).drop_duplicates(subset=["ID_Aluno","Momento"], keep="first")

# Merge (Left Join)
df = emb.merge(llm, on=["ID_Aluno","Momento"], how="left", suffixes=("", "_llmdup"))
df = df.drop(columns=[c for c in df.columns if c.endswith("_llmdup")])

print(f"Shape inicial pós-merge: {df.shape}")

# 3. FILTRAGEM: APENAS IDOSOS COM OS 4 MOMENTOS (LONGITUDINAL COMPLETO)
print("--- [PASSO 2] A FILTRAR PACIENTES COMPLETOS (MOMENTOS 1, 2, 3, 4) ---")

# Agrupar por ID e recolher o set de momentos únicos
valid_ids = []
required_moments = {1, 2, 3, 4}

for pid, group in df.groupby("ID_Aluno"):
    moments_present = set(group["Momento"].dropna().unique())
    if moments_present == required_moments:
        valid_ids.append(pid)

# Filtrar o DataFrame principal
df_clean = df[df["ID_Aluno"].isin(valid_ids)].copy()

# Ordenar para facilitar leitura (ID, depois Momento)
df_clean = df_clean.sort_values(by=["ID_Aluno", "Momento"]).reset_index(drop=True)

print(f"Pacientes iniciais: {df['ID_Aluno'].nunique()}")
print(f"Pacientes mantidos (com 4 rastreios): {len(valid_ids)}")
print(f"Shape final: {df_clean.shape}")

# 4. CARACTERIZAÇÃO DE VARIÁVEIS (TYPE INFERENCE)
print("--- [PASSO 3] CARACTERIZAÇÃO AUTOMÁTICA DE VARIÁVEIS ---")

def categorize_feature(series):
    # 1. Se todos os valores forem nulos
    if series.isnull().all():
        return "Empty"
    
    n_unique = series.nunique(dropna=True)
    
    # 2. Binário: Apenas 2 valores únicos (ex: 0/1, Sim/Não, M/F)
    if n_unique == 2:
        return "Binary"
    
    # 3. Numérico
    if pd.api.types.is_numeric_dtype(series):
        # Opcional: Se for numérico mas tiver poucos valores (ex: 1,2,3), pode ser Ordinal/Categórico
        # Mas para estatística, geralmente tratamos como Numérico Discreto
        return "Numeric"
    
    # 4. Categórico vs Texto Livre
    # Regra heurística: Se o número de valores únicos for baixo (<15) é Categoria.
    # Se for alto, provavelmente é Texto Livre ou ID.
    if n_unique < 15:
        return "Categorical"
    else:
        return "Text/ID"

# Criar dicionário de tipos
feat_types = {}
for col in df_clean.columns:
    feat_types[col] = categorize_feature(df_clean[col])

# Converter para DataFrame para visualização bonita
df_types = pd.DataFrame(list(feat_types.items()), columns=["Variavel", "Tipo_Inferido"])

# Contagem dos tipos
summary = df_types["Tipo_Inferido"].value_counts()

print("\nRESUMO DOS TIPOS DE VARIÁVEIS:")
print(summary)

print("\nEXEMPLO DETALHADO (PRIMEIRAS 20 VARIÁVEIS):")
print(df_types.head(20))

# 5. SALVAR OU EXPORTAR
# Se quiseres separar listas para usar depois:
numeric_vars = df_types[df_types["Tipo_Inferido"] == "Numeric"]["Variavel"].tolist()
binary_vars = df_types[df_types["Tipo_Inferido"] == "Binary"]["Variavel"].tolist()

print(f"\nVariáveis prontas para linregress (Numéricas): {len(numeric_vars)}")
# print(numeric_vars)

--- [PASSO 1] A CARREGAR E A FUNDIR DADOS ---
Shape inicial pós-merge: (6705, 115)
--- [PASSO 2] A FILTRAR PACIENTES COMPLETOS (MOMENTOS 1, 2, 3, 4) ---
Pacientes iniciais: 2844
Pacientes mantidos (com 4 rastreios): 506
Shape final: (2024, 115)
--- [PASSO 3] CARACTERIZAÇÃO AUTOMÁTICA DE VARIÁVEIS ---

RESUMO DOS TIPOS DE VARIÁVEIS:
Tipo_Inferido
Numeric        94
Binary         14
Text/ID         6
Categorical     1
Name: count, dtype: int64

EXEMPLO DETALHADO (PRIMEIRAS 20 VARIÁVEIS):
                                         Variavel Tipo_Inferido
0                                        ID_Aluno       Numeric
1                                         Momento       Numeric
2                              Local_de_Avaliacao       Numeric
3                                  Data_avaliacao   Categorical
4                                 Data_nascimento       Text/ID
5                                             Age       Numeric
6                                          Genero        Bina

In [ ]:
# REUSABLE CORRELATION ANALYSIS FUNCTION
import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr

def run_target_correlations(df, target_col, all_targets_list):
    """
    Calculates Pearson and Spearman correlations for a specific target,
    automatically dropping other potential targets to prevent leakage.
    """
    print(f"\n{'='*80}")
    print(f"🎯 ANALYZING TARGET: {target_col}")
    print(f"{'='*80}")

    # 1. Drop OTHER targets (Leakage prevention)
    targets_to_drop = [t for t in all_targets_list if t != target_col]
    df_corr = df.drop(columns=targets_to_drop, errors='ignore').copy()
    print(f"🗑️ Dropped competing targets: {targets_to_drop}")

    # 2. Check if Target Exists
    if target_col not in df_corr.columns:
        print(f"❌ Error: Target '{target_col}' not found in dataframe.")
        return None

    # 3. Clean Target (NaNs and Encoding)
    # Drop rows where target is unknown
    df_corr = df_corr.dropna(subset=[target_col])
    
    # Convert "Sim/Não" or "Yes/No" to 1/0 if needed
    if df_corr[target_col].dtype == 'object':
        clean_map = {'sim': 1, 'yes': 1, '1': 1, 1: 1, 'não': 0, 'nao': 0, 'no': 0, '0': 0, 0: 0}
        df_corr[target_col] = df_corr[target_col].astype(str).str.lower().map(clean_map)
    
    # Force numeric
    df_corr[target_col] = pd.to_numeric(df_corr[target_col], errors='coerce')
    
    print(f"✅ Data Ready. Valid Rows: {len(df_corr)}")

    # 4. Select Features (Numeric Only)
    feature_cols = df_corr.select_dtypes(include=[np.number]).columns.tolist()
    if target_col in feature_cols: feature_cols.remove(target_col)

    # 5. Run Correlations
    results = []
    for feature in feature_cols:
        # Get valid pairs (drop NaNs for this specific pair)
        temp_df = df_corr[[feature, target_col]].dropna()

        # Skip if not enough data (<10 rows) or constant value (no variance)
        if len(temp_df) < 10 or temp_df[feature].nunique() <= 1:
            continue

        # Calculate Statistics
        p_r, p_p = pearsonr(temp_df[feature], temp_df[target_col])
        s_r, s_p = spearmanr(temp_df[feature], temp_df[target_col])

        results.append({
            "Feature": feature,
            "Spearman_r": round(s_r, 4),
            "Spearman_p": round(s_p, 5),
            "Pearson_r": round(p_r, 4),
            "Pearson_p": round(p_p, 5),  # Added Pearson p-value here!
            "N": len(temp_df)
        })

    # 6. Format and Display
    if results:
        res_df = pd.DataFrame(results)
        
        # Sort by absolute Spearman strength
        res_df["Abs_S"] = res_df["Spearman_r"].abs()
        res_df = res_df.sort_values(by="Abs_S", ascending=False).drop(columns=["Abs_S"])

        # Add Significance Marker
        res_df["Signif"] = np.where(res_df["Spearman_p"] < 0.05, "*", "")

        # Print Table
        pd.set_option('display.max_rows', None)
        print(res_df.to_string(index=False))
        print("\nNote: '*' indicates Spearman p-value < 0.05")
        return res_df
    else:
        print("No valid correlations found.")
        return pd.DataFrame()

# EXECUTION BLOCK

# 1. Define your list of 3 targets
my_targets = [
    "Quedas_ultimos_12meses",
    "Quedas_12meses_Assistencia_hospitalar_Sim_Não",
    "Quedas_12meses_Hospitalizado_Sim_Não"
]

# 2. Call the function for each one
# Assuming 'df' is your loaded dataframe
corr_falls = run_target_correlations(df, my_targets[0], my_targets)



🎯 ANALYZING TARGET: Quedas_ultimos_12meses
🗑️ Dropped competing targets: ['Quedas_12meses_Assistencia_hospitalar_Sim_Não', 'Quedas_12meses_Hospitalizado_Sim_Não']
✅ Data Ready. Valid Rows: 6705
                                          Feature  Spearman_r  Spearman_p  Pearson_r  Pearson_p    N Signif
                           Quedas_12meses_Quantas      0.9913     0.00000     0.6654    0.00000 6705      *
                    Quedas_12meses_Situacao_score      0.9890     0.00000     0.9436    0.00000 6705      *
                                   Q12M_Sit_Emb_0     -0.9889     0.00000    -0.9236    0.00000 6705      *
                                   Q12M_Sit_Emb_1      0.8162     0.00000     0.8323    0.00000 6705      *
                     Quedas_12meses_Lesao_Sim_Nao      0.6941     0.00000     0.6941    0.00000 6705      *
                               Q12M_LesTipo_Emb_0     -0.6921     0.00000    -0.6167    0.00000 6705      *
                              Q12M_LesLocal_Emb_0

In [3]:
#Assistencia Hospitalar
corr_assist = run_target_correlations(df, my_targets[1], my_targets)


🎯 ANALYZING TARGET: Quedas_12meses_Assistencia_hospitalar_Sim_Não
🗑️ Dropped competing targets: ['Quedas_ultimos_12meses', 'Quedas_12meses_Hospitalizado_Sim_Não']
✅ Data Ready. Valid Rows: 6705
                                          Feature  Spearman_r  Spearman_p  Pearson_r  Pearson_p    N Signif
                   Quedas_12meses_Lesao_Gravidade      0.6592     0.00000     0.7362    0.00000 6705      *
                     Quedas_12meses_Lesao_Sim_Nao      0.6562     0.00000     0.6562    0.00000 6705      *
                              Q12M_LesLocal_Emb_0     -0.6549     0.00000    -0.6017    0.00000 6705      *
             Quedas_12meses_Lesao_Local_Gravidade      0.6466     0.00000     0.6110    0.00000 6705      *
                               Q12M_LesTipo_Emb_0     -0.6341     0.00000    -0.4572    0.00000 6705      *
                              Q12M_LesLocal_Emb_1      0.6250     0.00000     0.5964    0.00000 6705      *
                               Q12M_LesTipo_Emb_3

-Handgrip mao dominante - It's not significant probably because it introduces noise, and random diseases, and injuries.

In [4]:
corr_hosp = run_target_correlations(df, my_targets[2], my_targets)


🎯 ANALYZING TARGET: Quedas_12meses_Hospitalizado_Sim_Não
🗑️ Dropped competing targets: ['Quedas_ultimos_12meses', 'Quedas_12meses_Assistencia_hospitalar_Sim_Não']
✅ Data Ready. Valid Rows: 6705
                                          Feature  Spearman_r  Spearman_p  Pearson_r  Pearson_p    N Signif
                Quedas_12meses_Hospitalizado_Dias      1.0000     0.00000     0.7354    0.00000 6705      *
                   Quedas_12meses_Lesao_Gravidade      0.2894     0.00000     0.3812    0.00000 6705      *
                     Quedas_12meses_Lesao_Sim_Nao      0.2702     0.00000     0.2702    0.00000 6705      *
                              Q12M_LesLocal_Emb_0     -0.2683     0.00000    -0.2353    0.00000 6705      *
             Quedas_12meses_Lesao_Local_Gravidade      0.2678     0.00000     0.2753    0.00000 6705      *
                               Q12M_LesTipo_Emb_0     -0.2545     0.00000    -0.1468    0.00000 6705      *
                              Q12M_LesLocal_Emb_1